# 02 · Train DistilBERT on SQuAD 2.0

Run this in Google Colab (Runtime → Change runtime type → **T4 GPU**). It has two parts:

**Part A, sections 1–5: the smoke test (task T3, done).** A very short training run: 50 steps on 1% of the data, under a minute. It doesn't produce a useful model. It proves that every part of training works (data, model, loss, checkpoints) before we spend hours on the full run.

**Part B, sections 6–8: the full training run (task T6).** 2 epochs on all of SQuAD 2.0, about 2.5 hours on a T4. Checkpoints go to Google Drive, the finished model goes to the Hugging Face Hub, and the last section scores it on the dev set.

For the full run you only need the setup cells (section 0) and then sections 6–8. You can skip Part A.

All the code is in `src/train.py` and `src/evaluate.py`. All the settings are in `configs/distilbert.yaml`, with the smoke-test changes in `configs/smoke.yaml`.

## 0. Setup
Clones the repo and installs the requirements in Colab. Safe to run more than once.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Davidfaleye360/AI-Question-Answering-System.git"
REPO_DIR = "/content/AI-Question-Answering-System"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import transformers, datasets, torch

print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)
if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError("transformers 5.x is still loaded. Use Runtime -> Restart session, then run the setup cell again.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then run the setup cell again.")
    print("The smoke test still works without one, just more slowly.")

## 1. The settings
`configs/smoke.yaml` lists only what the smoke test changes. Everything else (learning rate, batch size, window size) is the same as the full run, so the smoke test exercises the real settings.

In [ ]:
from src.config import load_config

full = load_config("configs/distilbert.yaml")
cfg = load_config("configs/distilbert.yaml", overrides="configs/smoke.yaml")

print(f"Data used:      {cfg.data.train_fraction:.0%} of the training set (full run: {full.data.train_fraction:.0%})")
print(f"Training steps: {cfg.training.max_steps} (full run: {full.training.num_epochs} epochs)")
print(f"Batch size:     {cfg.training.batch_size} windows per step")
print(f"Learning rate:  {cfg.training.learning_rate}")
print(f"Loss recorded every {cfg.training.logging_steps} steps, checkpoint every {cfg.training.save_steps} steps")
print(f"Saving to:      {cfg.training.output_dir}")

## 2. Train for 50 steps

**What to expect.** The model has to pick the answer's start token and end token out of 384 positions. Before training it is guessing, and the loss for a blind guess among 384 options is about ln(384) ≈ 5.95. As it learns, the loss should fall well below that.

You'll also see a warning that some weights were "newly initialized". That's expected: the pretrained DistilBERT has no question-answering layer, so a new one is added and trained here.

`--fresh` starts from step 0 every time. Without it, the script continues from the newest checkpoint in the output folder, which is what we want for the long run in T6.

In [ ]:
!{sys.executable} -m src.train --config configs/distilbert.yaml --overrides configs/smoke.yaml --fresh

The last line must say **RESULT: PASS**: the loss went down, checkpoints were saved, and the final model was saved.

## 3. The loss curve

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

summary = json.load(open(f"{cfg.training.output_dir}/train_summary.json"))
losses = pd.DataFrame(summary["losses"], columns=["step", "loss"]).set_index("step")

ax = losses["loss"].plot(marker="o", color="#2a78d6", figsize=(7, 3.5))
ax.set_title(f"Training loss, smoke test ({summary['steps']} steps on {summary['device']})")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(f"First recorded loss: {losses['loss'].iloc[0]:.3f} | last: {losses['loss'].iloc[-1]:.3f} "
      f"| training time: {summary['runtime_seconds']:.0f}s")
print("Loss went down:", summary["loss_went_down"], "| RESULT:", "PASS" if summary["passed"] else "FAIL")

Each point is the average loss over the previous few steps. It jumps around from batch to batch, so look at the trend, not single points.

## 4. What was saved
Each `checkpoint-N` folder holds the model weights plus the optimizer state at step N, which is what lets a run continue after a disconnect. The files at the top level are the final model and tokenizer.

In [ ]:
from pathlib import Path

out = Path(cfg.training.output_dir)
for path in sorted(out.iterdir()):
    if path.is_dir():
        size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
        print(f"{path.name + '/':28} {size / 1e6:8.1f} MB")
    else:
        print(f"{path.name:28} {path.stat().st_size / 1e6:8.1f} MB")

## 5. Load the saved model
This checks that the saved files really are a usable model. After only 50 steps the answers are usually wrong or empty: the point here is that it loads and runs, not that it's right.

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering", model=cfg.training.output_dir, device=0 if torch.cuda.is_available() else -1)
qa(question="When did Curie win the Chemistry prize?",
   context="Marie Curie won the Nobel Prize in Physics in 1903 and in Chemistry in 1911.",
   handle_impossible_answer=True)

That is the end of the smoke test. It passed in Colab on 2026-11-10 (loss 5.86 → 3.58), so T3 is ticked.

---
# Part B: the full training run (T6)

## 6. Before you start

1. **GPU:** the second cell of this notebook must have printed `GPU: Tesla T4`.
2. **Hugging Face token:** click the **key icon** in Colab's left sidebar, add a secret named `HF_TOKEN` holding a token with **Write** permission, and switch on "Notebook access". Never paste the token into a cell: this repo is public.
3. **Google Drive:** about 2 GB free. The next cell asks for permission to connect your Drive. Use the same Google account every time.
4. **Keep this browser tab open** while it trains. Free Colab disconnects idle sessions.

**If Colab disconnects:** reconnect, run the two setup cells in section 0 again, then run the cells in this section again. Training continues from the newest checkpoint on your Drive instead of starting over.

The next cell connects Drive and reads the token.

In [ ]:
import shutil

RUN_DIR = "/content/drive/MyDrive/qa-project/distilbert-squad2"   # checkpoints and the final model go here

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # src/train.py reads the token from here
else:
    RUN_DIR = "outputs/distilbert-squad2"                # not in Colab: save inside the repo folder

os.makedirs(RUN_DIR, exist_ok=True)
free_gb = shutil.disk_usage(RUN_DIR).free / 1e9
print(f"Saving to: {RUN_DIR}")
print(f"Free space there: {free_gb:.1f} GB (the run needs about 2)")
print("Hugging Face token found:", bool(os.environ.get("HF_TOKEN")))
print("Checkpoints already there:", sorted(p for p in os.listdir(RUN_DIR) if p.startswith("checkpoint-")) or "none (a new run)")

## 7. Train

This is the same script as the smoke test, without the smoke-test overrides, so it uses everything in `configs/distilbert.yaml`: all 130,319 training questions, 2 epochs, about 16,500 steps.

**What to expect**
- First it checks that it can upload to the Hugging Face Hub. A wrong or read-only token fails here, in seconds, not after the training.
- The loss is printed every 100 steps. It starts near 5.9 and should fall to somewhere around 1 by the end.
- A checkpoint is saved to Drive every 2,000 steps (about every 20 minutes). Only the newest two are kept. Deleted ones go to your Drive trash, which still counts as used space: empty the trash if Drive fills up.
- At the end it prints **RESULT: PASS** and `Uploaded to https://huggingface.co/...`.

There is no `--fresh` here on purpose: running this cell again continues an interrupted run.

In [ ]:
!{sys.executable} -m src.train --config configs/distilbert.yaml --output-dir "{RUN_DIR}" --push-to-hub

The loss curve of the full run. Single values jump around, so the dark line is the average of the last 10 recorded values.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

run = json.load(open(f"{RUN_DIR}/train_summary.json"))
curve = pd.DataFrame(run["losses"], columns=["step", "loss"]).set_index("step")

ax = curve["loss"].plot(color="#2a78d6", alpha=0.35, figsize=(8, 3.5), label="every 100 steps")
curve["loss"].rolling(10, min_periods=1).mean().plot(ax=ax, color="#2a78d6", label="average of the last 10")
ax.set_title(f"Training loss, full run ({run['steps']:,} steps on {run['device']})")
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/loss_curve.png", dpi=150)
plt.show()

print(f"Steps: {run['steps']:,} | first loss: {curve['loss'].iloc[0]:.3f} | last: {curve['loss'].iloc[-1]:.3f} "
      f"| time of the last session: {run['runtime_seconds'] / 60:.0f} min | RESULT:", "PASS" if run["passed"] else "FAIL")

## 8. Score it on the dev set

Runs the trained model on all 11,873 dev questions (about a minute on a T4) and prints EM and F1, overall and for answerable and unanswerable questions. Our target is **65+ F1**. For comparison, RoBERTa scores 79.90 EM / 82.91 F1 with the same script.

In [ ]:
!{sys.executable} -m src.evaluate --model "{RUN_DIR}" --name distilbert-squad2 --output-dir "{RUN_DIR}/results"

## Next
- Copy the output of sections 7 (from `Trained ... steps` down to `Uploaded to ...`) and 8 (the score table) and send it to whoever is keeping the repo up to date. Then T6 gets ticked in TASKS.md and the scores go into the README.
- The model is now at the address printed by section 7. Anyone can load it with `pipeline("question-answering", model="...")`.
- The loss curve was saved as `loss_curve.png` in the Drive folder, for the report and the slides.